# Logistic Regression

Mục tiêu: dự đoán sinh viên rút lui sau ngày 28 từ dữ liệu ngày 0–27 (lớp dương là `label_withdrawn` = 1).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.models.evaluate import append_metrics_row, metrics_positive, pick_threshold
from src.models.train_lr import build_model, fit_and_score, load_preprocessor, load_split_data

FEATURE_PATH = ROOT / "data/processed/feature_28d.parquet"
SPLIT_PATH = ROOT / "data/processed/split.csv"
PREPROCESSOR_PATH = ROOT / "models/preprocessor.joblib"
METRICS_PATH = ROOT / "results/model_metrics.csv"
RANDOM_STATE = 42

In [ ]:
data = load_split_data(FEATURE_PATH, SPLIT_PATH)
X_train, y_train = data["train"]
X_dev, y_dev = data["dev"]
X_test, y_test = data["test"]

In [ ]:
preprocessor = load_preprocessor(PREPROCESSOR_PATH)
X_train_t = preprocessor.transform(X_train)
X_dev_t = preprocessor.transform(X_dev)
X_test_t = preprocessor.transform(X_test)

In [ ]:
model = build_model()
dev_score = fit_and_score(model, X_train_t, y_train, X_dev_t)

In [ ]:
threshold = pick_threshold(y_dev, dev_score, metric="f1")
dev_metrics = metrics_positive(y_dev, dev_score, threshold)
append_metrics_row(METRICS_PATH, "logistic_regression", "dev", dev_metrics)
dev_metrics

Tập test chỉ chạy một lần ở cuối, sau khi đã khóa mô hình và ngưỡng.

In [ ]:
test_score = model.predict_proba(X_test_t)[:, 1]
test_metrics = metrics_positive(y_test, test_score, threshold)
append_metrics_row(METRICS_PATH, "logistic_regression", "test", test_metrics)
test_metrics